# eXERCICE 1

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Chargement du dataset depuis l'URL publique (évite les chemins locaux non portables)
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic_data = pd.read_csv(url)

In [ ]:
# Correction : utiliser titanic_data et non data_ex1 (variable non définie)
print(titanic_data.duplicated().sum())

# The result is 0 so there is not duplicated rows

# Exerice 2

In [ ]:
titanic_data.isnull().sum()

# NOUS Avons donc 177 manques de données au niveau de la colonne age, 687 manques dans la colonne Cabin et 2 autre dans Embarked

In [ ]:
titanic_data = titanic_data.drop(columns=['Cabin'])

In [ ]:
titanic_data

In [ ]:
pip install scikit-learn

In [ ]:
from sklearn.impute import KNNImputer

In [ ]:
imputer = KNNImputer(n_neighbors=2)
impute_data = imputer.fit_transform(titanic_data[["Pclass", "Age", "Fare"]])

In [ ]:
impute_data

In [ ]:
titanic_data["Age"] = impute_data[:, 1]
# gérer les ages manquants

In [ ]:
titanic_data

In [ ]:
titanic_data.isnull().sum()

In [ ]:
# gérer les embarked manquants

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="most_frequent")

impute_data = imputer.fit_transform(titanic_data[["Embarked"]])
impute_data = pd.DataFrame(impute_data, columns=["Embarked"])
impute_data

In [ ]:
titanic_data["Embarked"] = impute_data["Embarked"]

# Exercice 3 : Ingénierie des fonctionnalités

In [ ]:
# -----------------------------
# FamilySize feature
# -----------------------------
# SibSp = nombre de frères/soeurs/conjoints
# Parch = nombre de parents/enfants
# +1 pour compter le passager lui-même

titanic_data["FamilySize"] = (
    titanic_data["SibSp"]
    + titanic_data["Parch"]
    + 1
)

# =========================================
# 2. EXTRACT TITLE FROM NAME
# =========================================

# Exemple :
# "Braund, Mr. Owen Harris"
# -> "Mr"

titanic_data["Title"] = (
    titanic_data["Name"]
    .str.split(",")
    .str[1]
    .str.split(".")
    .str[0]
    .str.strip()
)

In [ ]:
# Encodage One-Hot pour Sex, Embarked et Title
# On utilise get_dummies une seule fois ici (pas de LabelEncoder qui serait contradictoire avec l'exercice 6)
titanic_data = pd.get_dummies(
    titanic_data,
    columns=["Sex", "Embarked", "Title"],
    drop_first=True
)

print(titanic_data.head())

# Exercice 4

On continue de travailler sur **titanic_data** déjà prétraité — pas de rechargement du CSV.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.boxplot(titanic_data["Fare"].dropna())
plt.title("Boxplot of Fare")
plt.show()

In [ ]:
plt.boxplot(titanic_data["Age"].dropna())
plt.title("Boxplot of Age")
plt.show()

In [ ]:
titanic_data["Fare"].hist(bins=30)
plt.title("Distribution of Fare")
plt.xlabel("Fare")
plt.ylabel("Frequency")
plt.show()

In [ ]:
titanic_data["Age"].hist(bins=30)
plt.title("Distribution of Age")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Détection avec IQR

Q1 = titanic_data["Fare"].quantile(0.25)
Q3 = titanic_data["Fare"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

fare_outliers = titanic_data[
    (titanic_data["Fare"] < lower_bound)
    |
    (titanic_data["Fare"] > upper_bound)
]

print(fare_outliers.shape)
print(fare_outliers[["Fare"]].head())

In [ ]:
# Méthode 1 : Quantile Capping
print(titanic_data["Fare"].quantile(0.98))
print(titanic_data["Fare"].quantile(0.99))

upper_cap = titanic_data["Fare"].quantile(0.98)

titanic_capped = titanic_data.copy()
titanic_capped["Fare"] = titanic_capped["Fare"].clip(upper=upper_cap)

In [ ]:
# Vérifier après traitement
plt.boxplot(titanic_capped["Fare"].dropna())
plt.title("Fare After Quantile Capping")
plt.show()

In [ ]:
# Méthode : Log Transformation

import numpy as np

titanic_log = titanic_data.copy()
titanic_log["Fare"] = np.log1p(titanic_log["Fare"])

In [ ]:
titanic_log["Fare"].hist(bins=30)
plt.title("Fare After Log Transformation")
plt.show()

In [ ]:
# Méthode : Row Removal

titanic_removed = titanic_data[
    (titanic_data["Fare"] >= lower_bound)
    &
    (titanic_data["Fare"] <= upper_bound)
]

In [ ]:
# Comparer le nombre de lignes
print("Before:", titanic_data.shape)
print("After removal:", titanic_removed.shape)

In [ ]:
plt.boxplot(titanic_data["Fare"].dropna())
plt.title("Before Treatment")
plt.show()

In [ ]:
plt.boxplot(titanic_capped["Fare"].dropna())
plt.title("After Quantile Capping")
plt.show()

In [ ]:
plt.boxplot(titanic_log["Fare"].dropna())
plt.title("After Log Transformation")
plt.show()

In [ ]:
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler
)

In [ ]:
# Création les scalers
standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

In [ ]:
# StandardScaler pour Age et FamilySize
# Le scaling est appliqué sur titanic_data après le traitement des outliers (exercice 4)

# Avant scaling
print(titanic_data[["Age", "FamilySize"]].head())

In [ ]:
# Scaling
titanic_data[["Age", "FamilySize"]] = (
    standard_scaler.fit_transform(
        titanic_data[["Age", "FamilySize"]]
    )
)

In [ ]:
# MinMaxScaler pour Fare
# On applique le scaling sur Fare après le traitement des outliers
print(titanic_data["Fare"].head())

In [ ]:
# Scaling
titanic_data[["Fare"]] = (
    minmax_scaler.fit_transform(
        titanic_data[["Fare"]]
    )
)

In [ ]:
# Après scaling
import matplotlib.pyplot as plt

titanic_data["Age"].hist(bins=30)
plt.title("Scaled Age")
plt.show()

In [ ]:
titanic_data["Fare"].hist(bins=30)
plt.title("Normalized Fare")
plt.show()

# Exercice 6

In [ ]:
# Display data types
print(titanic_data.dtypes)

In [ ]:
categorical_columns = (
    titanic_data
    .select_dtypes(include=["object"])
    .columns
)

print(categorical_columns)
# Sex, Embarked et Title ont déjà été encodés en exercice 3
# Aucune colonne catégorielle restante à encoder ici

In [ ]:
titanic_data.head()

In [ ]:
# Exercise 7: Data Transformation for Age Feature

In [ ]:
# =========================================
# CREATE AGE GROUPS
# =========================================

titanic_data["AgeGroup"] = pd.cut(
    titanic_data["Age"],
    bins=[-3, -0.5, 0, 1, 4],
    labels=[
        "Child",
        "Teen",
        "Adult",
        "Senior"
    ]
)

# =========================================
# CHECK AGE GROUPS
# =========================================

print(
    titanic_data[
        ["Age", "AgeGroup"]
    ].head()
)

# =========================================
# ENCODE AGE GROUPS
# =========================================

titanic_data = pd.get_dummies(
    titanic_data,
    columns=["AgeGroup"],
    drop_first=True
)

# =========================================
# FINAL CHECK
# =========================================

print(titanic_data.head())